### ingest_daily_support_tickets

In [1]:
!pip install pymysql



[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: C:\Users\DELL\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [2]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: C:\Users\DELL\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [3]:
pip install boto3

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: C:\Users\DELL\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [4]:
pip install sqlalchemy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: C:\Users\DELL\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [5]:
pip install python-dotenv


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: C:\Users\DELL\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [6]:
import os
import pandas as pd
import boto3
from io import StringIO
from sqlalchemy import create_engine
from datetime import datetime, timedelta

from dotenv import load_dotenv

load_dotenv()

False

In [7]:
os.getenv("REGION")

In [8]:
# ---------- CONFIG ----------
from getpass import getpass

db_config = {
    "user": "root",              # Keep just the username here
    "password": "WJ25@kmph", 
    "host": "localhost",         # 💡 Change this to just "localhost" or "127.0.0.1"
    "port": 3306,                # Make sure your MySQL port is correct (usually 3306)
    "database": "newdataproject"
}


S3_BUCKET = "an.09-ni" 
S3_PREFIX = "support-tickets/raw/"  
DATE_TRACKER_FILE = "date_tracker.txt"

import os

AWS_CONFIG = {
    "aws_access_key_id": getpass("Access key ID: ").strip(),
    "aws_secret_access_key": getpass("Secret access key: ").strip(),
    "region_name": "ap-south-1"   # change to your bucket's region
}

Access key ID:  ········
Secret access key:  ········


In [9]:
print(repr(db_config["host"]))
print(repr(db_config["user"]))
print("@ in password:", "@" in db_config["password"])

'localhost'
'root'
@ in password: True


In [10]:
# ---------- UTILITY FUNCTIONS ----------
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

def get_engine(config):
    url = URL.create(
        drivername="mysql+pymysql",
        username=config["user"],
        password=config["password"],
        host=config["host"],
        port=int(config["port"]),
        database=config["database"],
    )
    return create_engine(url)
def upload_to_s3(df, bucket, key):
    bucket = bucket.strip()          # <-- add this line
    csv_buffer = StringIO()
    df.to_csv(csv_buffer, index=False)

    s3 = boto3.client('s3', **AWS_CONFIG)
    s3.put_object(Bucket=bucket, Key=key, Body=csv_buffer.getvalue())
    print(f"✅ Uploaded to s3://{bucket}/{key}")

def read_last_date(file_path):
    if os.path.exists(file_path):
        with open(file_path, 'r') as f:
            return f.read().strip()
    return "2025-06-30"  # Starting point before 1st July

def update_last_date(file_path, new_date):
    with open(file_path, 'w') as f:
        f.write(new_date)

def get_next_date(last_date_str):
    last_date = datetime.strptime(last_date_str, "%Y-%m-%d")
    next_date = last_date + timedelta(days=1)
    return next_date.strftime("%Y-%m-%d")

# ---------- MAIN INGESTION LOGIC ----------
def run_ingestion():
    engine = get_engine(db_config)
    last_date = read_last_date(DATE_TRACKER_FILE)
    next_date = get_next_date(last_date)

    # Query only that day’s data
    query = f"""
        SELECT * FROM careplus_support_tickets
        WHERE DATE(created_at) = '{next_date}';
    """
    df = pd.read_sql(query, engine)
    print(df.shape)
    print(df.head())

    if df.empty:
        print(f"⚠️ No data found for {next_date}. Skipping upload.")
        return

    # Upload to S3
    s3_key = f"{S3_PREFIX}support_tickets_{next_date}.csv"
    upload_to_s3(df, S3_BUCKET, s3_key)

    # Update date tracker
    update_last_date(DATE_TRACKER_FILE, next_date)
    print(f"📅 Updated tracker to {next_date}")

# Run
if __name__ == "__main__":
    run_ingestion()

(33, 10)
  ï»¿ticket_id        created_at       resolved_at   agent priority  \
0   TCK0712000  2025-07-12 00:37  2025-07-12 10:10   Sneha      Low   
1   TCK0712001  2025-07-12 00:42  2025-07-12 14:00   Rohit   Medium   
2   TCK0712002  2025-07-12 00:59  2025-07-13 05:13  Ananya      Hgh   
3   TCK0712003  2025-07-12 01:11  2025-07-12 02:52  Ananya    Medum   
4   TCK0712004  2025-07-12 01:34  2025-07-12 13:34  Ananya   Medium   

   num_interactions   issue_category   channel    status agent_feedback  
0                 8       Bug Report     Phone  Resolved                 
1           -999999   Account Locked     Phone  Resolved                 
2                 8       Bug Report     Phone  Resolved                 
3                 4       Bug Report  Web Form  Resolved                 
4                 7  Feature Request  Web Form  Resolved                 
✅ Uploaded to s3://an.09-ni/support-tickets/raw/support_tickets_2025-07-12.csv
📅 Updated tracker to 2025-07-12


In [11]:
from sqlalchemy import text

engine = get_engine(db_config)
with engine.connect() as conn:
    print(conn.execute(text("SELECT 1")).scalar())

1


In [12]:
print(type(AWS_CONFIG))
print(list(AWS_CONFIG.keys()))
for k, v in AWS_CONFIG.items():
    print(k, "->", "empty" if not v else f"set (length {len(str(v))})")

<class 'dict'>
['aws_access_key_id', 'aws_secret_access_key', 'region_name']
aws_access_key_id -> set (length 20)
aws_secret_access_key -> set (length 40)
region_name -> set (length 10)
